# 🦈🦌🐆 Matrix Compression with SVD

### ⏱️ The whole story in 30 seconds
1. 📱 **The problem:** photos take up a lot of space.
2. 🔢 **The key fact:** to a computer, a photo is just a big table of numbers (a *matrix*).
3. 🍰 **The trick:** a math tool called **SVD** splits that table into *layers*,
   from most important to least important.
4. ✂️ **The result:** keep only the important layers, throw away the rest —
   with **5% of the data** you can already recognise a shark 🦈

### 🗺️ The process at a glance

| 📷 | ➡️ | 🔢 | ➡️ | 🍰 | ➡️ | ✂️ | ➡️ | 🖼️ |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **Photo** | | **Table of numbers** | | **Split into layers** (SVD) | | **Keep the top k layers** | | **Smaller photo** |

> 💬 You don't need to read the code (the grey boxes). Just read the text and look at the pictures 👀

▶️ To run everything: click **Run All** at the top.

In [ ]:
# 🧰 Setup: load the tools we need (you can skip reading this)
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

IMAGE_DIR = Path("images")  # input photos
OUT_DIR = Path("image")     # figures for the slides
OUT_DIR.mkdir(exist_ok=True)

ANIMALS = {"Shark": "shark.jpg", "Deer": "deer.jpg", "Leopard": "leopard.jpg"}
COLORS = {"Shark": "#2a78d6", "Deer": "#eb6834", "Leopard": "#1baf7a"}
LINESTYLES = {"Shark": "-", "Deer": "--", "Leopard": ":"}

plt.rcParams.update({
    "font.size": 13, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e3e3e0", "axes.edgecolor": "#52514e",
    "figure.facecolor": "white",
})


def save(fig, name):
    """Save a figure into image/ (for Canva) and show it here."""
    fig.savefig(OUT_DIR / name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 1. 📱 The problem: photos are big

Every photo on your phone is made of tiny coloured dots called **pixels**.
Our shark photo is 1000 × 673 pixels — that's **673,000 dots**, and each dot needs
3 numbers (red, green, blue) 🔴🟢🔵. That's about **2 million numbers** for one photo!

> 🗣️ **In plain words:** more numbers = bigger file = slower to send and more storage used.
> Can we keep the photo looking good but store **fewer numbers**? 🤔

## 2. 🔢 To a computer, a photo is just numbers

Let's zoom into a tiny 8 × 8 square of the shark photo 🔍
Each little square is one pixel, and the number is how bright it is:
**0 = black ⚫, 255 = white ⚪**.

In [ ]:
shark = np.asarray(Image.open(IMAGE_DIR / "shark.jpg").convert("RGB"), dtype=float)
gray = shark.mean(axis=2)          # brightness only, to keep it simple
top, left, size = 300, 520, 8      # where the little square is
patch = gray[top:top + size, left:left + size]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1.5, 1]})
ax1.imshow(shark.astype(np.uint8))
ax1.add_patch(plt.Rectangle((left, top), size, size, fill=False, color="red", lw=3))
ax1.annotate("we zoom in here", (left + size, top), xytext=(left + 120, top - 120),
             color="red", fontsize=15, arrowprops={"color": "red", "lw": 2})
ax1.set_title("The photo", fontsize=16)
ax1.axis("off")

ax2.imshow(patch, cmap="gray", vmin=0, vmax=255)
for r in range(size):
    for c in range(size):
        value = int(patch[r, c])
        ax2.text(c, r, value, ha="center", va="center", fontsize=11,
                 color="white" if value < 128 else "black")
ax2.set_title("Zoomed in: just a table of numbers", fontsize=16)
ax2.axis("off")
fig.tight_layout()
save(fig, "00a_photo_is_numbers.png")

A **colour** photo is simply **3 of these tables** stacked together: one for red 🔴,
one for green 🟢, one for blue 🔵. Mix them and you get the colours you see.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, (c, name, cmap) in zip(axes, [(0, "Red", "Reds_r"), (1, "Green", "Greens_r"),
                                      (2, "Blue", "Blues_r")]):
    ax.imshow(shark[:, :, c], cmap=cmap, vmin=0, vmax=255)
    ax.set_title(f"{name} table", fontsize=16)
    ax.axis("off")
fig.suptitle("A colour photo = 3 tables of numbers", fontsize=18)
fig.tight_layout()
save(fig, "00b_three_colour_tables.png")

> 🗣️ **In plain words:** in math, a table of numbers is called a **matrix**.
> So "compressing a photo" really means "compressing a matrix" — that's our project! 🎯

## 3. 🧩 The trick: patterns can be written shorter

Look at this little table. Do you notice something? 👀

|   |   |   |
|---|---|---|
| 2 | 4 | 6 |
| 1 | 2 | 3 |
| 3 | 6 | 9 |

Every row is just `1 2 3` multiplied by something (×2, ×1, ×3).
So instead of writing **9 numbers**, we can write **6**:

- the pattern: `1 2 3`
- how much of it each row uses: `2, 1, 3`

> 🗣️ **In plain words:** it's like a recipe 📝 — instead of listing every cookie,
> you write the recipe once and say "make 2 batches, 1 batch, 3 batches".

A math tool called **SVD** (Singular Value Decomposition) finds these patterns automatically 🪄
Let's check it on our little table:

In [ ]:
A = np.array([[2, 4, 6],
              [1, 2, 3],
              [3, 6, 9]], dtype=float)

U, S, Vt = np.linalg.svd(A)
print("How important each pattern is:", S.round(3))
print("-> only ONE pattern matters, the others are 0\n")

one_layer = S[0] * np.outer(U[:, 0], Vt[0])
print("Rebuilt from that one pattern:")
print(one_layer.round(3))

✅ One pattern rebuilds the whole table perfectly.

## 4. 🍰 SVD splits a photo into layers

A real photo isn't one perfect pattern — but SVD can split it into **many layers**,
sorted from **most important** to **least important**. Think of painting a picture 🎨:

- 🖌️ **first layers:** rough shapes — where it's dark, where it's light
- ✏️ **middle layers:** outlines — the shark's body, fins and tail
- 🔍 **last layers:** tiny details — teeth, bubbles, skin texture

Top row below: single layers on their own. Bottom row: adding them up — watch the shark appear! 🦈

In [ ]:
U, S, Vt = np.linalg.svd(gray, full_matrices=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for col, i in enumerate([1, 2, 3, 10]):
    layer = S[i - 1] * np.outer(U[:, i - 1], Vt[i - 1])
    axes[0, col].imshow(layer, cmap="gray")
    axes[0, col].set_title(f"layer {i} only")
    total = U[:, :i] @ np.diag(S[:i]) @ Vt[:i, :]
    axes[1, col].imshow(np.clip(total, 0, 255), cmap="gray", vmin=0, vmax=255)
    axes[1, col].set_title(f"layers 1 to {i} added")
for ax in axes.flat:
    ax.axis("off")
fig.suptitle("SVD splits a picture into layers, most important first", fontsize=18)
fig.tight_layout()
save(fig, "01_svd_layers.png")

> 🗣️ **In plain words:** with just 10 layers you can already guess it's a shark.
> Each layer is cheap to store, so keeping a few layers instead of the whole photo saves a lot of space 💾

## 5. ✂️ Compressing: keep only the first **k** layers

**k** is simply **how many layers we keep** — think of it as a quality dial 🎛️

- 🔻 small k → tiny file, blurry photo
- 🔺 big k → sharp photo, bigger file

We measure two things:

- 💾 **Storage:** how much space we use, compared with the original (100%).
- 🎯 **Quality score (PSNR):** how close it looks to the original, in "dB".
  Rule of thumb: **below 20 = bad 😖, 25 = okay 🙂, above 30 = your eye can barely tell 😍**

The code below does the compressing (you can skip reading it):

In [ ]:
def load(filename):
    """Load a photo as 3 tables of numbers and split each one into layers (SVD)."""
    img = np.asarray(Image.open(IMAGE_DIR / filename).convert("RGB"), dtype=float)
    svds = [np.linalg.svd(img[:, :, c], full_matrices=False) for c in range(3)]
    return img, svds


def compress(svds, k):
    """Rebuild the photo from only the first k layers of each colour."""
    channels = [U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :] for U, S, Vt in svds]
    return np.clip(np.stack(channels, axis=2), 0, 255).astype(np.uint8)


def storage(img, k):
    """Fraction of the original storage: each layer costs m + n + 1 numbers."""
    m, n, _ = img.shape
    return k * (m + n + 1) / (m * n)


def psnr(img, approx):
    """Quality score in dB: 10 log10(255^2 / average squared error)."""
    return 10 * np.log10(255 ** 2 / np.mean((img - approx) ** 2))


def break_even(img):
    """Largest k that still uses less storage than the original."""
    m, n, _ = img.shape
    return m * n // (m + n + 1)


data = {name: load(filename) for name, filename in ANIMALS.items()}
print("✅ All 3 photos are split into layers and ready!")

## 6. 🎮 Try it yourself!

Change the animal and the number of layers **K**, then run the cell again (⌨️ Shift + Enter).

💡 Good values to try: **K = 5** (very blurry), **K = 20** (recognisable), **K = 200** (almost perfect).

In [ ]:
ANIMAL = "Shark"  # 🦈 "Shark", 🦌 "Deer" or 🐆 "Leopard"
K = 20            # how many layers to keep

img, svds = data[ANIMAL]
approx = compress(svds, K)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.8))
axes[0].imshow(img.astype(np.uint8))
axes[0].set_title("Original (100% storage)", fontsize=16)
axes[1].imshow(approx)
axes[1].set_title(f"k = {K}:  {storage(img, K):.1%} storage · quality {psnr(img, approx):.1f} dB",
                  fontsize=16)
for ax in axes:
    ax.axis("off")
fig.tight_layout()
plt.show()
plt.close(fig)

## 7. 🖼️ Results: more layers = sharper photo, bigger file

For each animal: the original next to k = 5, 20, 50, 100 and 200 layers.
Under each picture you can see **how much space it uses** and its **quality score**.

In [ ]:
K_GRID = [5, 20, 50, 100, 200]

for name, (img, svds) in data.items():
    fig, axes = plt.subplots(2, 3, figsize=(16, 8.6), gridspec_kw={"hspace": 0.35})
    axes[0, 0].imshow(img.astype(np.uint8))
    axes[0, 0].set_title("Original\n100% storage", fontsize=15)
    for ax, k in zip(axes.flat[1:], K_GRID):
        approx = compress(svds, k)
        ax.imshow(approx)
        ax.set_title(f"k = {k}\n{storage(img, k):.1%} storage · {psnr(img, approx):.1f} dB",
                     fontsize=15)
    for ax in axes.flat:
        ax.axis("off")
    fig.suptitle(f"{name}: keeping k layers", fontsize=19)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.88, bottom=0.01, wspace=0.04)
    save(fig, f"02_grid_{name.lower()}.png")

> 🗣️ **In plain words:**
> - 🦈 at **k = 20** we use only **5%** of the space and the shark is already recognisable.
> - 🦌 the deer looks good sooner than the others — its sky and blurry background are smooth and easy to store.
> - 🐆 the leopard's spots and the shark's bubbles 🫧 are busy details, so they need more layers.

## 8. 🔍 Small details come last

Let's zoom into the shark's teeth 🦷 and the leopard's face.
The big shapes appear early, but the fine details only show up with many layers.

In [ ]:
ZOOMS = {  # (top, bottom, left, right) as fractions of the picture
    "Shark": (0.38, 0.70, 0.40, 0.60),
    "Leopard": (0.26, 0.62, 0.38, 0.58),
}

for name, (y0, y1, x0, x1) in ZOOMS.items():
    img, svds = data[name]
    m, n, _ = img.shape
    rows, cols = slice(int(y0 * m), int(y1 * m)), slice(int(x0 * n), int(x1 * n))
    panels = [(f"k = {k}", compress(svds, k)) for k in [20, 50, 200]]
    panels.append(("Original", img.astype(np.uint8)))

    fig, axes = plt.subplots(1, 4, figsize=(16, 5))
    for ax, (title, picture) in zip(axes, panels):
        ax.imshow(picture[rows, cols])
        ax.set_title(title, fontsize=15)
        ax.axis("off")
    fig.suptitle(f"{name} close-up: fine details appear last", fontsize=18)
    fig.tight_layout()
    save(fig, f"03_zoom_{name.lower()}.png")

> 🗣️ **In plain words:** SVD doesn't know what a shark is 🤷 — it keeps whatever covers the most of
> the picture first (like the big blue sea 🌊). The teeth are small, so they come last,
> even though *we* think they're the most important part!

## 9. 📉 Why does it work?

Each layer has an **importance score** (mathematicians call it a *singular value*).
The chart shows these scores for all three photos.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6.5))
for name, (img, svds) in data.items():
    S = np.mean([s for _, s, _ in svds], axis=0)  # average of R, G, B
    ax.semilogy(np.arange(1, len(S) + 1), S / S[0], color=COLORS[name],
                ls=LINESTYLES[name], lw=2.5, label=name)
ax.set(xlim=(1, 400), ylim=(1e-3, 1.3), xlabel="layer number",
       ylabel="importance (compared with layer 1)",
       title="Importance of each layer drops quickly")
ax.legend(frameon=False)
fig.tight_layout()
save(fig, "04_singular_values.png")

> 🗣️ **In plain words:** the first few layers are **hundreds of times** more important than the later ones 📉
> So throwing away the later layers loses very little — that's why compression works! 🎉

## 10. 🤔 How many layers are worth it?

This chart shows **quality** (up = better) against **space used** (left = smaller file).
The best spot is **high up and far left** 👆👈

In [ ]:
K_CURVE = list(range(1, 20)) + list(range(20, 401, 10))

fig, ax = plt.subplots(figsize=(12, 6.5))
for name, (img, svds) in data.items():
    x = [storage(img, k) * 100 for k in K_CURVE]
    y = [psnr(img, compress(svds, k)) for k in K_CURVE]
    ax.plot(x, y, color=COLORS[name], ls=LINESTYLES[name], lw=2.5, label=name)
ax.axhline(30, color="#52514e", ls="--", lw=1)
ax.text(1, 30.4, "30 dB ≈ looks very good", color="#52514e")
ax.set(xlim=(0, 100), xlabel="storage used (% of original)",
       ylabel="quality, PSNR (dB)", title="Quality vs. storage")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
save(fig, "05_quality_vs_storage.png")

for name, (img, _) in data.items():
    print(f"{name:8} saves space only up to k = {break_even(img)} layers")

> 🗣️ **In plain words:**
> - 🦌 the deer reaches "looks very good" (30 dB) using only about **25%** of the space.
> - 🦈🐆 the shark and leopard need about **45%**.
> - ⚠️ Careful: after about **400 layers**, the "compressed" version is actually **bigger** than the original,
>   because each layer costs a little space too!

In [ ]:
K_TABLE = [5, 20, 50, 100, 200]

rows = []
for name, (img, svds) in data.items():
    rows.append([name] + [f"{psnr(img, compress(svds, k)):.1f} dB" for k in K_TABLE])
img = data["Shark"][0]
rows.append(["Storage (shark)"] + [f"{storage(img, k):.1%}" for k in K_TABLE])

fig, ax = plt.subplots(figsize=(12, 3))
ax.axis("off")
table = ax.table(cellText=rows, colLabels=["Image"] + [f"k = {k}" for k in K_TABLE],
                 loc="center", cellLoc="center")
table.scale(1, 2.2)
table.set_fontsize(14)
for (r, _), cell in table.get_celld().items():
    cell.set_edgecolor("#d8d8d4")
    if r == 0:
        cell.set_text_props(weight="bold")
        cell.set_facecolor("#f0efec")
ax.set_title("Quality (PSNR) for each k", fontsize=16)
save(fig, "06_results_table.png")

## 11. 🤓 Bonus for math fans: is this really the *best* we can do?

Yes! A theorem called **Eckart–Young** proves that keeping the first $k$ layers of the SVD
is the **best possible** way to approximate a matrix with $k$ layers 🏆

$$A_k = \sum_{i=1}^{k} \sigma_i u_i v_i^T, \qquad
\|A - A_k\|_F = \sqrt{\sigma_{k+1}^2 + \sigma_{k+2}^2 + \cdots}$$

The second formula says: the error is exactly the size of the layers we threw away 🗑️
Let's check it with real numbers:

In [ ]:
img, svds = data["Shark"]
A = img[:, :, 0]          # red table of the shark
U, S, Vt = svds[0]

print(f"{'k':>4} {'actual error':>14} {'predicted by the theorem':>27}")
for k in [10, 50, 200]:
    A_k = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    print(f"{k:>4} {np.linalg.norm(A - A_k):>14.2f} {np.sqrt(np.sum(S[k:] ** 2)):>27.2f}")
print("\n✅ They match exactly!")

## 12. 🏁 Summary

1. 🔢 A photo is a **matrix** — a big table of numbers.
2. 🍰 **SVD** splits it into **layers**, from most to least important.
3. ✂️ Keeping only the first **k** layers gives a much smaller file that still looks like the photo.
4. 🦈 With **5%** of the space the shark is recognisable; with about **45%** it looks very good.
5. 🌄 Smooth photos (🦌 deer) compress better than busy ones (🦈 bubbles 🫧, 🐆 spots).
6. 🏆 Math (Eckart–Young) proves this is the best possible choice of k layers.
7. ⚠️ **Limitation:** SVD keeps what covers the most of the picture, not what *people* find
   important — small details like teeth 🦷 come last.

🎉 **Thanks for reading!**